# Open-Source AI Human-Avatar Generation Worker (Kaggle Accelerator)

**Track 02 — Open-Source AI Human-Avatar Generation**  
**Assessment**: INCUBRIX PRIVATE LIMITED — SASTRA 2027 Graduate Hiring  
**Execution Environment**: Kaggle Notebook (Free GPU Accelerator: NVIDIA T4 / P100)  

---

### Purpose & Operational Contract
This notebook serves as the **Free Accelerator Execution Environment** for the Avatar System. It consumes portable job bundles prepared by the local laptop orchestrator, runs deterministic image generation using an openly available, pinned open-weights model (`stable-diffusion-v1-5/stable-diffusion-v1-5`), captures hardware telemetry, calculates SHA-256 checksums, writes provenance manifests (`avatar_manifest.json`), and packages outputs into `outputs.zip` for local ingestion.

**No proprietary or paid APIs are used.**

In [ ]:
# Step 1: Install pinned dependencies
!pip install --quiet --upgrade pip
!pip install --quiet \
    "diffusers==0.32.2" \
    "transformers==4.49.0" \
    "accelerate>=0.28.0" \
    "safetensors>=0.4.2" \
    "pillow>=10.2.0" \
    "pydantic>=2.7.0,<3.0.0" \
    "pyyaml>=6.0.1" \
    "psutil>=5.9.8"

In [ ]:
# Step 2: System Telemetry & Accelerator Verification
import os
import sys
import time
import json
import platform
import hashlib
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image
import torch
import psutil

print("=" * 60)
print("SYSTEM TELEMETRY & HARDWARE PROVENANCE")
print("=" * 60)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"Platform:         {platform.platform()}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:       {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No CUDA device detected. Running on CPU (reduced speed).")
print("=" * 60)

In [ ]:
# Step 3: Define Helper Functions for Manifest & File Integrity
def compute_sha256(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

def get_software_versions() -> dict:
    import diffusers
    import transformers
    import accelerate
    return {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "diffusers": diffusers.__version__,
        "transformers": transformers.__version__,
        "accelerate": accelerate.__version__,
        "cuda": torch.version.cuda if torch.cuda.is_available() else "none",
    }

In [ ]:
# Step 4: Load Pinned Open Diffusion Pipeline
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MODEL_REVISION = "main"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Loading pinned model: {MODEL_REPO} (Revision: {MODEL_REVISION}) on {DEVICE}...")
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_REPO,
    revision=MODEL_REVISION,
    torch_dtype=DTYPE,
    safety_checker=None,  # We execute our own pre-generation multi-rule safety engine
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)

# Enable memory optimizations if on GPU
if DEVICE == "cuda":
    pipe.enable_attention_slicing()
    print("Attention slicing enabled. Pipeline ready.")

In [ ]:
# Step 5: Input Discovery (Scan for Job Bundles or create Baseline Bundles)
INPUT_JOBS_DIR = Path("/kaggle/input/avatar-jobs")
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

job_dirs = []
if INPUT_JOBS_DIR.exists():
    job_dirs = [p for p in INPUT_JOBS_DIR.iterdir() if p.is_dir() and (p / "job.json").exists()]

if not job_dirs:
    print("No input job bundles found in /kaggle/input/avatar-jobs. Checking local ./jobs directory...")
    local_jobs = Path("./jobs")
    if local_jobs.exists():
        job_dirs = [p for p in local_jobs.iterdir() if p.is_dir() and (p / "job.json").exists()]

print(f"Discovered {len(job_dirs)} job bundle(s) to process.")

In [ ]:
# Step 6: Execute Deterministic Inference & Provenance Tracking
results_log = []
sw_versions = get_software_versions()

for job_dir in job_dirs:
    with open(job_dir / "job.json", "r", encoding="utf-8") as f:
        job_data = json.load(f)

    job_id = job_data["job_id"]
    avatar_spec = job_data["avatar_spec"]
    avatar_id = avatar_spec["avatar_id"]
    prompts = job_data["prompts"]
    seed = job_data["seed"]
    metadata = job_data["metadata"]
    safety_res = job_data.get("safety_result", {"is_safe": True, "refusal_reasons": []})

    # Verify pre-generation safety
    if not safety_res.get("is_safe", True):
        print(f"[REFUSED] Job {job_id} was flagged as unsafe. Skipping generation.")
        continue

    width = metadata.get("width", 512)
    height = metadata.get("height", 512)
    steps = metadata.get("inference_steps", 25)
    guidance = metadata.get("guidance_scale", 7.5)

    print(f"\nExecuting Job: {job_id} | Avatar: {avatar_id} | Seed: {seed} | Resolution: {width}x{height}...")
    
    # Reset peak memory stats
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    
    t_start = time.time()
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    # Diffusion Inference
    out = pipe(
        prompt=prompts["positive_prompt"],
        negative_prompt=prompts["negative_prompt"],
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
    )
    duration = time.time() - t_start
    peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024**2)) if DEVICE == "cuda" else 0.0

    # Save Image
    image = out.images[0]
    img_filename = f"{avatar_id}_{seed}.png"
    img_path = OUTPUT_DIR / img_filename
    image.save(img_path, format="PNG")
    sha256_hash = compute_sha256(img_path)

    # Construct Machine-Readable Provenance Manifest
    manifest = {
        "manifest_id": f"man_{job_id}_{int(time.time())}",
        "job_id": job_id,
        "avatar_id": avatar_id,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "status": "success",
        "avatar_spec": avatar_spec,
        "positive_prompt": prompts["positive_prompt"],
        "negative_prompt": prompts["negative_prompt"],
        "seed": seed,
        "model_name": MODEL_REPO,
        "model_revision": MODEL_REVISION,
        "sampler_or_scheduler": "DPMSolverMultistepScheduler",
        "inference_steps": steps,
        "guidance_scale": guidance,
        "image_width": width,
        "image_height": height,
        "aspect_ratio": avatar_spec.get("aspect_ratio", "1:1"),
        "output_filename": img_filename,
        "image_sha256": sha256_hash,
        "synthetic_media": True,
        "synthetic_label": "AI-Generated Fictional Human Avatar",
        "software_versions": sw_versions,
        "compute_route": "kaggle_accelerator",
        "safety_result": safety_res,
        "execution_telemetry": {
            "duration_seconds": round(duration, 3),
            "peak_vram_mb": round(peak_vram_mb, 1),
            "device_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
            "torch_version": torch.__version__,
        }
    }

    manifest_path = OUTPUT_DIR / f"{avatar_id}_{seed}_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(f"[COMPLETE] Saved {img_filename} (SHA256: {sha256_hash[:12]}...) in {duration:.2f}s | VRAM: {peak_vram_mb:.1f} MB")
    results_log.append(manifest)

print("=" * 60)
print(f"BATCH EXECUTION COMPLETE: Successfully processed {len(results_log)} avatar jobs.")
print("=" * 60)

In [ ]:
# Step 7: Export Outputs into Downloadable Portable ZIP Archive
zip_output_path = Path("/kaggle/working/outputs.zip")
with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, file_path.name)

print(f"Exported portable archive: {zip_output_path} (Size: {zip_output_path.stat().st_size / (1024*1024):.2f} MB)")
print("Download outputs.zip and run locally: `avatar ingest --results outputs.zip`")